# Retail Demand Forecasting
## Data Cleaning and Initial EDA

This notebook focuses on understanding and preparing the retail sales data
and performing an initial exploratory data analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = "../data/raw/"

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")
stores = pd.read_csv(DATA_PATH + "stores.csv")
transactions = pd.read_csv(DATA_PATH + "transactions.csv")
holidays = pd.read_csv(DATA_PATH + "holidays_events.csv")
oil = pd.read_csv(DATA_PATH + "oil.csv")

In [ ]:
datasets = {
    "train": train,
    "test": test,
    "stores": stores,
    "transactions": transactions,
    "holidays": holidays,
    "oil": oil
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(df.head())

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(df.dtypes)

In [ ]:
date_columns = {
    "train": train,
    "test": test,
    "transactions": transactions,
    "holidays": holidays,
    "oil": oil
}

for name, df in date_columns.items():
    df["date"] = pd.to_datetime(df["date"])

In [ ]:
train["date"].min(), train["date"].max()

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    display(missing)

In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicates")

In [ ]:
print("Rows:", len(train))
print("Stores:", train["store_nbr"].nunique())
print("Product Families:", train["family"].nunique())
print("Start Date:", train["date"].min())
print("End Date:", train["date"].max())

In [ ]:
duplicate_keys = train.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

print("Duplicate date-store-family combinations:", duplicate_keys)

In [ ]:
train["sales"].describe()

In [ ]:
print("Zero sales:", (train["sales"] == 0).sum())
print(
    "Zero sales percentage:",
    round((train["sales"] == 0).mean() * 100, 2),
    "%"
)

In [ ]:
print("Negative sales:", (train["sales"] < 0).sum())

In [ ]:
print(
    "Rows with promotion:",
    round((train["onpromotion"] > 0).mean() * 100, 2),
    "%"
)

In [ ]:
oil[oil["dcoilwtico"].isna()]

In [ ]:
oil_missing_dates = oil.loc[
    oil["dcoilwtico"].isna(), "date"
]

holidays[holidays["date"].isin(oil_missing_dates)][
    ["date", "type", "locale", "description"]
]

In [ ]:
oil["dcoilwtico"] = oil["dcoilwtico"].ffill().bfill()

### Handling Missing Oil Prices

The oil dataset contains missing values in the `dcoilwtico` column, which represents the daily oil price.

During the inspection, we found 43 missing oil-price values. Several of the missing dates also appear in the holidays dataset, but not all missing dates were verified as holidays.

Since oil price is a time-dependent feature, we will not remove these rows. Instead, missing values will be imputed using forward fill, with backward fill used only when no previous value is available.

This approach preserves the dates and avoids removing observations that may be important for demand analysis.

In [ ]:
oil["dcoilwtico"].isna().sum()

After imputation, we verify that no missing oil-price values remain.

### Duplicate Check

We check for duplicate rows across all datasets to ensure that duplicated observations do not affect the analysis or lead to incorrect aggregations during data integration.

In [ ]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

### Train Data Granularity Check

The training data is expected to contain one observation for each combination of date, store, and product family. We therefore check whether any duplicate combinations exist.

In [ ]:
train.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

### Date Validation

Since this project focuses on time-series demand forecasting, we verify that the date columns are correctly formatted and inspect the date ranges across the datasets.

In [ ]:
for name, df in datasets.items():
    if "date" in df.columns:
        print(f"{name}: {df['date'].dtype}")

In [ ]:
for name, df in datasets.items():
    if "date" in df.columns:
        print(
            f"{name}: "
            f"{df['date'].min().date()} → "
            f"{df['date'].max().date()}"
        )

In [ ]:
print(
    "Train unique dates:",
    train["date"].nunique()
)

print(
    "Train total rows:",
    len(train)
)

In [ ]:
print(
    "Train date range:",
    train["date"].max() - train["date"].min()
)

In [ ]:
expected_dates = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="D"
)

actual_dates = train["date"].drop_duplicates().sort_values()

missing_dates = expected_dates.difference(actual_dates)

print("Expected dates:", len(expected_dates))
print("Actual dates:", len(actual_dates))
print("Missing dates:", len(missing_dates))

In [ ]:
print("Missing dates:")
for date in missing_dates:
    print(date.date())

### Missing Calendar Dates

The training data contains four missing calendar dates:

- December 25, 2013
- December 25, 2014
- December 25, 2015
- December 25, 2016

All four missing dates correspond to December 25. These dates are not individual missing sales values; the entire dates are absent from the training dataset.

Therefore, we will not impute or add sales observations for these dates. Adding artificial sales values could introduce assumptions that are not supported by the data.

### Decision

The missing dates will be kept as absent dates during the initial analysis. 
The forecasting models will use the available observations without creating artificial demand values for these dates.

### Sales Validation

Sales is the target variable for the demand forecasting task. We inspect its distribution, range, zero values, and negative values to identify potential data quality issues before performing exploratory analysis.

In [ ]:
print("Sales summary:")
display(train["sales"].describe())

In [ ]:
print("Zero sales:", (train["sales"] == 0).sum())
print("Zero sales percentage:", round((train["sales"] == 0).mean() * 100, 2), "%")
print("Negative sales:", (train["sales"] < 0).sum())

### Sales Validation Result

The sales variable contains no negative values, indicating no obvious invalid negative demand records.

However, 31.3% of the observations have zero sales. These zero values are retained because they represent potentially meaningful demand patterns and will be considered during the analysis of regular and intermittent demand.

## Initial Exploratory Data Analysis

The initial EDA examines overall sales trends and patterns over time, followed by differences across product families, stores, months, weekdays, and promotion status.

In [ ]:
daily_sales = (
    train.groupby("date", as_index=False)["sales"]
    .sum()
)

plt.figure(figsize=(14, 5))

plt.plot(
    daily_sales["date"],
    daily_sales["sales"]
)

plt.title("Total Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Total Sales")

plt.tight_layout()
plt.show()

### Insight

The overall daily sales show an increasing trend over the observed period, with sales levels generally higher in later years than in the earlier years.

The series also shows strong day-to-day variability and recurring patterns over time, suggesting the presence of temporal seasonality.

Several dates show very low or zero total sales, which is consistent with the high proportion of zero-sales observations identified during data validation. These observations are retained because they may represent meaningful demand patterns.

### 1. Overall Sales Trend with 7-Day Rolling Average

A 7-day rolling average is used to smooth daily fluctuations and provide a clearer view of the underlying sales trend.

In [ ]:
daily_sales["rolling_7d"] = (
    daily_sales["sales"]
    .rolling(window=7)
    .mean()
)

plt.figure(figsize=(14, 5))

plt.plot(
    daily_sales["date"],
    daily_sales["rolling_7d"]
)

plt.title("7-Day Rolling Average of Total Daily Sales")
plt.xlabel("Date")
plt.ylabel("7-Day Average Sales")

plt.tight_layout()
plt.show()

### Insight

The 7-day rolling average shows an overall upward trend in sales over the observed period.

Sales levels are generally higher in 2016 and 2017 compared with the earlier years. The series also contains recurring fluctuations, suggesting that temporal patterns and seasonality may influence demand.

A noticeable decline occurs during part of 2015, followed by a recovery and a higher sales level in the later period.

### 3. Top 10 Product Families by Total Sales

We aggregate total sales by product family to identify the product categories that contribute most to overall demand.

In [ ]:
family_sales = (
    train.groupby("family")["sales"]
    .sum()
    .sort_values(ascending=False)
)

top_10_families = family_sales.head(10)

plt.figure(figsize=(12, 6))

top_10_families.sort_values().plot(kind="barh")

plt.title("Top 10 Product Families by Total Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product Family")

plt.tight_layout()
plt.show()

### Insight

GROCERY I has the highest total sales among all product families, followed by BEVERAGES and PRODUCE.

The difference between the top product family and the others is substantial, indicating that demand is not evenly distributed across product families.

This suggests that product family should be considered as an important feature when building the demand forecasting model.

### 4. Sales by Promotion Status

We compare total sales on days with and without promotions to examine whether promotional activity is associated with higher demand.

In [ ]:
promotion_sales = (
    train.groupby(train["onpromotion"] > 0)["sales"]
    .sum()
)

promotion_sales.index = ["No Promotion", "Promotion"]

plt.figure(figsize=(8, 5))

promotion_sales.plot(kind="bar")

plt.title("Total Sales by Promotion Status")
plt.xlabel("Promotion Status")
plt.ylabel("Total Sales")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

### Promotion Sales Validation

Total sales are higher for observations with promotions. However, total sales alone may be affected by the different number of observations in each group.

Therefore, we compare the average sales per observation to better understand the relationship between promotion activity and demand.

In [ ]:
promotion_summary = (
    train.groupby(train["onpromotion"] > 0)["sales"]
    .agg(["count", "mean", "median", "sum"])
)

promotion_summary.index = ["No Promotion", "Promotion"]

display(promotion_summary)

In [ ]:
no_promo_mean = promotion_summary.loc["No Promotion", "mean"]
promo_mean = promotion_summary.loc["Promotion", "mean"]

percentage_difference = (
    (promo_mean - no_promo_mean) / no_promo_mean
) * 100

print(
    "Average sales difference:",
    round(percentage_difference, 2),
    "%"
)

### Insight

Observations with promotions have substantially higher average sales than observations without promotions.

The average sales are 1,137.69 for promoted observations compared with 158.25 for non-promoted observations, representing a difference of approximately 618.94%.

This strong association suggests that promotion activity may be an important predictive feature for demand forecasting. However, this analysis does not establish a causal relationship between promotions and sales.

### 5. Average Sales by Month

We examine the average sales across months to identify recurring monthly patterns and potential seasonal effects in demand.

In [ ]:
monthly_sales = (
    train.groupby(train["date"].dt.month)["sales"]
    .mean()
)

plt.figure(figsize=(10, 5))

monthly_sales.plot(kind="bar")

plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

### Insight

Average sales vary across months, indicating a recurring monthly pattern in demand.

December has the highest average sales, while February has the lowest. Although the differences between most months are moderate, the monthly variation suggests that month-related seasonal features may be useful for demand forecasting.

## Advanced EDA & Feature Exploration

The following sections extend the initial exploratory analysis with calendar-based patterns, store-level differences, and feature preparation for demand forecasting.

### 6. Sales by Day of Week

We examine average sales across days of the week to identify whether demand follows a recurring weekly pattern.

In [ ]:
train['weekday'] = train['date'].dt.day_name()

weekday_sales = train.groupby('weekday')['sales'].mean()

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 
                  'Friday', 'Saturday', 'Sunday']
weekday_sales = weekday_sales.reindex(weekday_order)

plt.figure(figsize=(9, 5))
weekday_sales.plot(kind='bar')
plt.title('Average Sales by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(weekday_sales.round(2))

### Insight

Sales are highest on weekends, with Sunday (463.09) and Saturday (433.34) showing the strongest average demand. Thursday records the lowest average sales (283.54), while the remaining weekdays (Monday through Friday) show relatively similar levels, ranging from 283.54 to 346.54.

This weekly pattern, with weekend sales roughly 1.6 times higher than the weakest weekday, suggests that day-of-week should be included as a categorical feature in the forecasting model to capture this recurring demand cycle.

### 7. Sales by Year (Seasonality)

We examine sales trends across years to identify year-over-year growth.

In [ ]:
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month

yearly_sales = train.groupby('year')['sales'].mean()

plt.figure(figsize=(9, 5))
yearly_sales.plot(kind='bar')
plt.title('Average Sales by Year')
plt.xlabel('Year')
plt.ylabel('Average Sales')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(yearly_sales.round(2))

### Verifying the Growth Trend

To verify whether the year-over-year growth reflects genuine demand increase or incomplete data tracking, we examine the PRODUCE family in isolation, since fresh categories are more likely to have been added to the system gradually.

In [ ]:
produce_by_year = train[train['family'] == 'PRODUCE'].groupby('year')['sales'].mean()
print(produce_by_year.round(2))

### Insight

Average sales show a clear upward trend from 2013 (216.48) to 2017 (480.12). However, this growth is partly an artifact of incomplete tracking in early years: the PRODUCE family, for example, shows average sales of only 3.70 in 2013 compared with 1,196.20 in 2014 — a 323-fold increase that reflects the category not being fully recorded early in the dataset, rather than genuine demand growth.

This confirms that year should be included as a feature with caution, and that per-family tracking start dates should be verified (similar to the store cold-start check) before treating year-over-year growth as a reliable trend signal.

### 8. Sales by Store

We examine sales distribution across individual stores to identify high- and low-performing locations.

In [ ]:
store_sales = train.groupby('store_nbr')['sales'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
store_sales.plot(kind='bar')
plt.title('Average Sales by Store')
plt.xlabel('Store Number')
plt.ylabel('Average Sales')
plt.tight_layout()
plt.show()

print(store_sales.head(5))
print("...")
print(store_sales.tail(5))

### Verifying Store Coverage

Before interpreting store-level averages, we verify whether all stores have equal data coverage across the period.

In [ ]:
store_days = train.groupby('store_nbr')['date'].nunique()

print("Days recorded per store — Store 52:", store_days.loc[52])
print("Average days across all stores:", store_days.mean().round(0))

### Insight

Average sales vary substantially across stores, from 1,117.25 (Store 44) to 48.52 (Store 52) — roughly a 23-fold difference. All stores have identical data coverage (1,684 recorded days), so the gap is not caused by missing records.

However, Store 52 opened partway through the data period, meaning its average includes an extended stretch of zero sales prior to opening rather than reflecting weak performance once operational. Store-level averages should therefore be interpreted alongside each store's operational start date, and a store-opening feature (or excluding pre-opening records) would improve the reliability of store comparisons.

### 9. Sales by Store Type

We examine whether store type, as defined in the stores metadata, is associated with different demand levels.

In [ ]:
stores = pd.read_csv(DATA_PATH + "stores.csv")
train_stores = train.merge(stores, on='store_nbr', how='left')

type_sales = train_stores.groupby('type')['sales'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
type_sales.plot(kind='bar')
plt.title('Average Sales by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Average Sales')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(type_sales.round(2))

### Insight

Store type is strongly associated with sales volume. Type A stores average 705.88 in sales, more than double the next highest type (D at 350.98), while Type C records the lowest average at 197.26 — a 3.6-fold gap between the highest and lowest types.

Notably, the ordering does not follow the alphabetical labels (D outperforms B, and E outperforms C), indicating that store type reflects an operational classification rather than a simple size ranking. This makes store type a useful categorical feature for the forecasting model, as it captures systematic differences in demand levels that are not evident from the store number alone.

### 10. Promotion Effect by Product Family

The initial analysis showed that promoted observations have substantially higher average sales overall. We now examine whether this effect is consistent across product families or concentrated in specific categories.

In [ ]:
promo_effect = train.groupby(['family', train['onpromotion'] > 0])['sales'].mean().unstack()
promo_effect.columns = ['No Promotion', 'Promotion']

promo_effect['lift_pct'] = (
    (promo_effect['Promotion'] - promo_effect['No Promotion']) 
    / promo_effect['No Promotion'] * 100
).round(1)

promo_effect = promo_effect.sort_values('lift_pct', ascending=False)

print(promo_effect.head(10))
print("...")
print(promo_effect.tail(10))

### Insight

The promotional effect varies dramatically across product families, ranging from a 4,257% lift (School and Office Supplies) to 23.9% (Prepared Foods) — a spread that the aggregate figure of 618.94% conceals entirely.

However, the largest percentage lifts occur in categories with very low baseline sales: School and Office Supplies averages 1.02 units without promotion versus 44.53 with, and Baby Care averages 0.11 versus 1.66. In absolute terms, the most commercially significant lift is in Produce, which rises from 792.10 to 2,438.22 — a 207.8% increase on an already substantial base.

Books shows no promotional data at all, indicating the category was never promoted across the entire period.

This suggests that promotion should be modelled as an interaction with product family rather than as a single global effect, since the response magnitude differs by two orders of magnitude between categories.

### 11. Holiday Effect on Sales

We examine whether sales differ on public holidays compared with regular days. The holidays dataset distinguishes several event types, including transferred holidays that are officially moved to another date.

In [ ]:
holidays = pd.read_csv(DATA_PATH + "holidays_events.csv")
holidays["date"] = pd.to_datetime(holidays["date"])

national_holidays = holidays[
    (holidays["locale"] == "National") & (holidays["transferred"] == False)
]["date"].unique()

daily_sales = train.groupby("date")["sales"].mean()

holiday_sales = daily_sales[daily_sales.index.isin(national_holidays)]
regular_sales = daily_sales[~daily_sales.index.isin(national_holidays)]

print("Holidays:", len(holiday_sales), "days | Mean sales:", round(holiday_sales.mean(), 2))
print("Regular:", len(regular_sales), "days | Mean sales:", round(regular_sales.mean(), 2))
print("Difference:", round((holiday_sales.mean() - regular_sales.mean()) / regular_sales.mean() * 100, 2), "%")

In [ ]:
np.random.seed(42)

random_diffs = []
for _ in range(10):
    random_days = np.random.choice(regular_sales.index, size=136, replace=False)
    random_mean = daily_sales[daily_sales.index.isin(random_days)].mean()
    others_mean = daily_sales[~daily_sales.index.isin(random_days)].mean()
    random_diffs.append((random_mean - others_mean) / others_mean * 100)

print("Holiday effect:", round((holiday_sales.mean() - regular_sales.mean()) / regular_sales.mean() * 100, 2), "%")
print("Random samples (10 runs):", [round(d, 2) for d in random_diffs])
print("Max random difference:", round(max(abs(d) for d in random_diffs), 2), "%")

### Insight

Sales on national holidays average 419.34 compared with 352.37 on regular days, a 19.01% increase across 136 holiday observations.

To confirm this is not an artifact of random variation, we compared the holiday effect against ten random samples of equal size (136 days) drawn from regular days. The random differences ranged from -6.02% to +3.44%, with a maximum absolute deviation of 6.02% — roughly one third of the observed holiday effect, and centred around zero as expected for noise.

This confirms that national holidays represent a genuine demand signal rather than random fluctuation, and should be included as a binary feature in the forecasting model. Transferred holidays were excluded from this analysis, as the dataset documentation indicates they behave as ordinary working days.

### 12. Merging Supplementary Data

We merge store metadata, daily transactions, and oil prices into the main dataset to prepare a unified table for feature exploration and modelling.

In [ ]:
transactions = pd.read_csv(DATA_PATH + "transactions.csv")
transactions["date"] = pd.to_datetime(transactions["date"])

oil = pd.read_csv(DATA_PATH + "oil.csv")
oil["date"] = pd.to_datetime(oil["date"])
oil["dcoilwtico"] = oil["dcoilwtico"].ffill().bfill()

df = train.merge(stores, on="store_nbr", how="left")
df = df.merge(transactions, on=["date", "store_nbr"], how="left")
df = df.merge(oil, on="date", how="left")

df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

print("Shape after merging:", df.shape)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

### Handling Missing Transactions

The transactions dataset does not record entries for days when a store had no activity. We verify this by examining sales on rows with missing transaction counts.

In [ ]:
missing_trans = df[df["transactions"].isna()]

print("Sales on rows with missing transactions:")
print(missing_trans["sales"].describe().round(2))
print("\nZero-sales percentage:", round((missing_trans["sales"] == 0).mean() * 100, 2), "%")

df["transactions"] = df["transactions"].fillna(0)

print("\nMissing values after filling:", df.isna().sum().sum())

### Insight

Transaction counts are missing for 245,784 rows (8.2% of the dataset). Sales on these rows are zero in 98.68% of cases, confirming that missing entries correspond to days when a store recorded no activity rather than to data collection errors.

We therefore fill these values with zero rather than imputing a mean or median, since imputing an average transaction count would introduce artificial activity on days when stores were closed or not yet operational.

### 13. Correlation Between Numerical Features

We examine the linear relationships between numerical variables to identify which features carry predictive signal for sales, and to detect any redundancy between them.

In [ ]:
numeric_cols = ["sales", "onpromotion", "transactions", "dcoilwtico", "cluster"]

corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

print(corr["sales"].sort_values(ascending=False).round(3))

### Insight

Promotion status shows the strongest linear relationship with sales (r = 0.428), consistent with the family-level analysis where promotional lift ranged from 24% to over 4,000% depending on category. Transaction count follows at r = 0.233, reflecting general store activity but not the monetary value of individual purchases.

Store cluster (r = 0.039) and oil price (r = -0.075) show negligible linear correlation with sales. The cluster result is expected, since cluster is a categorical identifier rather than an ordered quantity, and linear correlation is not a meaningful measure for it.

The weak oil-price correlation is notable given that the dataset documentation highlights Ecuador's oil dependence. This suggests that any macroeconomic effect operates over longer horizons than daily sales variation, and that oil price is unlikely to be a useful direct feature at the daily store-family level.

### 14. Impact of the April 2016 Earthquake

The dataset documentation notes that a magnitude 7.8 earthquake struck Ecuador on 16 April 2016, and that relief efforts significantly affected supermarket sales in the following weeks. We examine whether this event is visible in the data and how it was distributed across stores.

In [ ]:
quake_period = train[train["date"].between("2016-04-01", "2016-05-15")]
daily_total = quake_period.groupby("date")["sales"].sum()

plt.figure(figsize=(13, 5))
plt.plot(daily_total.index, daily_total.values, marker="o", markersize=4)
plt.axvline(pd.Timestamp("2016-04-16"), color="red", linestyle="--", label="Earthquake (16 April 2016)")
plt.title("Total Daily Sales Around the April 2016 Earthquake")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Isolating the Earthquake Effect from Weekly Seasonality

The sales increase after the earthquake coincides partly with a normal weekend peak. To separate the two, we compare each day against the average for the same weekday during undisturbed periods before and after the event.

In [ ]:
train["weekday_num"] = train["date"].dt.dayofweek

baseline_period = train[
    train["date"].between("2016-03-01", "2016-04-10")
    | train["date"].between("2016-05-16", "2016-06-15")
]

baseline_by_weekday = baseline_period.groupby("weekday_num")["sales"].sum() / \
                       baseline_period.groupby("weekday_num")["date"].nunique()

quake_week = train[train["date"].between("2016-04-16", "2016-04-24")]
quake_daily = quake_week.groupby("date").agg(
    total_sales=("sales", "sum"),
    weekday_num=("weekday_num", "first")
)

quake_daily["expected"] = quake_daily["weekday_num"].map(baseline_by_weekday)
quake_daily["lift_pct"] = (
    (quake_daily["total_sales"] - quake_daily["expected"]) / quake_daily["expected"] * 100
).round(1)

print(quake_daily[["total_sales", "expected", "lift_pct"]].round(0).to_string())

### Insight

The earthquake produced a clear but bounded effect on sales. After adjusting for weekly seasonality by comparing each day against the same weekday in undisturbed periods, the pattern is:

- **16 April (day of the earthquake):** −7% — sales fell below the expected level, consistent with the event occurring in the evening and disrupting normal shopping.
- **17–21 April:** sustained elevation, peaking at +88% on 18 April, reflecting relief-driven purchasing of water and essential goods.
- **22–24 April:** rapid decay to +19%, +10%, and +3%, returning to baseline within roughly one week.

The raw sales chart alone would overstate this effect, since the post-earthquake days coincided with a normal weekend peak. The seasonality-adjusted figures isolate the genuine impact.

For modelling purposes, this period represents a non-recurring shock that no forecasting model can anticipate. We will flag 17–23 April 2016 with a binary indicator so the models can account for it rather than treating it as regular demand variation.

### 15. Regular vs Intermittent Demand Patterns

A central premise of this project is that product families differ fundamentally in how their demand behaves: some sell consistently every day, while others sell sporadically with long gaps between transactions. A single forecasting approach is unlikely to serve both patterns equally well.

We quantify this by measuring the proportion of zero-sales days for each product family.

In [ ]:
zero_pct = (
    train.groupby("family")["sales"]
    .apply(lambda s: (s == 0).mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

print(zero_pct.to_string())

### Separating Structural from Genuine Zeros

The lowest zero-sales rate is identical (8.06%) across several staple families, which indicates a shared structural floor rather than genuine demand behaviour. We investigate this before drawing conclusions about demand patterns.

In [ ]:
zeros_per_date = train.groupby("date")["sales"].apply(lambda s: (s == 0).mean() * 100)

print("Dates with highest zero-sales rate:")
print(zeros_per_date.sort_values(ascending=False).head(10).round(2).to_string())

In [ ]:
train_clean = train[~((train["date"].dt.month == 1) & (train["date"].dt.day == 1))].copy()

first_sale = (
    train_clean[train_clean["sales"] > 0]
    .groupby(["store_nbr", "family"])["date"].min()
    .rename("first_sale_date")
)

train_clean = train_clean.merge(first_sale, on=["store_nbr", "family"], how="inner")
train_clean = train_clean[train_clean["date"] >= train_clean["first_sale_date"]]

zero_pct_clean = (
    train_clean.groupby("family")["sales"]
    .apply(lambda s: (s == 0).mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

print(zero_pct_clean.to_string())

In [ ]:
def classify_demand(pct):
    if pct >= 50:
        return "Intermittent"
    elif pct >= 20:
        return "Irregular"
    else:
        return "Regular"

demand_pattern = zero_pct_clean.to_frame("zero_pct")
demand_pattern["pattern"] = demand_pattern["zero_pct"].apply(classify_demand)

print(demand_pattern["pattern"].value_counts())
print()
print(demand_pattern.to_string())

### Visualising the Two Extremes

To illustrate the difference in demand behaviour, we compare a representative regular family with a representative intermittent family over the same period.

In [ ]:
period = ("2016-01-01", "2016-03-31")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

configs = [
    (1, "BEVERAGES", "Regular demand (0.48% zero days)"),
    (15, "BABY CARE", "Intermittent demand (82.13% zero days)")
]

for ax, (store, fam, label) in zip(axes, configs):
    series = train_clean[
        (train_clean["store_nbr"] == store)
        & (train_clean["family"] == fam)
        & (train_clean["date"].between(*period))
    ].set_index("date")["sales"]

    ax.plot(series.index, series.values, marker="o", markersize=3)
    ax.set_title(f"{fam} (Store {store}) — {label}")
    ax.set_ylabel("Sales")
    ax.grid(alpha=0.3)

axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

### Insight

Product families differ fundamentally in demand behaviour. After removing structural zeros (New Year closures and pre-launch periods for each store-family combination), zero-sales rates range from 0.48% for staples such as Beverages, Dairy and Bread to 82.13% for Baby Care.

The cleaning step is essential: raw zero-sales rates showed an identical 8.06% floor across all staple families, which reflected shared closure days rather than genuine demand behaviour. After correction, Books dropped from 96.96% to 65.03% and Baby Care from 94.13% to 82.13%, and the ranking of the most intermittent families changed.

We classify families into three groups based on the cleaned zero-sales rate:

- **Intermittent** (≥50% zero days): Baby Care, Home Appliances, Books, School and Office Supplies — 4 families
- **Irregular** (20–50%): Hardware, Magazines, Pet Supplies, Lawn and Garden, Ladieswear, Players and Electronics, Celebration, Beauty — 8 families
- **Regular** (<20%): 21 families, including all daily staples

The contrast is visible in the two series plotted above: Beverages shows a continuous line with strong weekly seasonality, while Baby Care alternates between zero and sporadic spikes with no stable pattern.

This confirms the core premise of the project. Standard time-series models rely on continuity and recent history, both of which are absent in intermittent series. Applying a single modelling strategy across all families would systematically underperform on the intermittent group, which justifies our plan to evaluate model performance separately by demand pattern.

### 16. Candidate Features for Forecasting

Based on the patterns identified above, we summarise the features most likely to carry predictive signal, along with the reasoning behind each.

**Strong candidates**

- `onpromotion` — the strongest single predictor (r = 0.428), with effects varying substantially by product family. Should be modelled as an interaction with family rather than a global effect.
- `family` — demand levels and promotional response differ by two orders of magnitude across categories.
- `day_of_week` — weekend sales are roughly 1.6 times higher than the weakest weekday.
- `is_holiday` — national holidays show a 19% uplift, confirmed against random-sample variation.
- `store_type` — a 3.6-fold gap between the highest and lowest performing types.
- `store_nbr` — substantial store-level variation beyond what type alone explains.
- `demand_pattern` — families classify into three groups by cleaned zero-sales rate (Regular <20%, Irregular 20–50%, Intermittent ≥50%). This distinction underpins the modelling strategy, since continuity-dependent methods are expected to underperform on the four intermittent families.

**Moderate candidates**

- `transactions` — captures general store activity (r = 0.233), though it reflects footfall rather than basket value.
- `month` — moderate seasonal variation, with December highest and February lowest.

**Weak or excluded**

- `dcoilwtico` — negligible correlation at the daily level (r = -0.075). Any macroeconomic effect appears to operate over longer horizons than daily demand variation.
- `year` — shows a strong upward trend, but this is partly an artifact of incomplete category tracking in early periods and should not be used as a direct trend feature without correction.
- `cluster` — a categorical identifier; linear correlation is not meaningful, though it may still hold value if encoded appropriately.

**Features to engineer**

Lag features (`lag_1`, `lag_7`, `lag_14`) and rolling statistics (`rolling_mean_7`, `rolling_mean_14`) are expected to be essential for time-series forecasting, capturing recent demand levels and short-term momentum that none of the raw columns provide.

An `earthquake_period` binary flag (17–23 April 2016) should also be included, to prevent the models from treating a one-off shock as recurring demand behaviour.

### 17. Forecasting Granularity

Based on the structure of the data and the patterns identified above, we define the forecasting granularity as **date × store × product family** — the same level at which the raw data is recorded.

This level is chosen because store-level and family-level effects are both substantial and independent: average sales vary 23-fold across stores and show a 3.6-fold gap across store types, while promotional response ranges from 24% to over 4,000% depending on product family. Aggregating to either store level or family level alone would discard signal that the other dimension carries.

Critically, the intermittent demand pattern identified above is only visible at this level of detail. Aggregating across stores would mask zero-sales days entirely, since the sum across 54 stores is rarely zero even for sparse families — eliminating the very signal the project aims to model.

This yields 1,782 individual time series (54 stores × 33 families), each forecast over a 15-day horizon.

### 18. Saving the Modeling Dataset

We save the merged and cleaned dataset for use in the feature engineering and modelling stages.

In [ ]:
import os

df = df.merge(
    demand_pattern.reset_index()[["family", "pattern"]],
    on="family",
    how="left"
)

df = df.drop(columns=["weekday_num"], errors="ignore")

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/modeling_dataset.csv", index=False)

print("Saved:", df.shape)
print("Columns:", list(df.columns))